# CS4207 Group 12: Predicting NYC yellow-taxi trip duration

We predict how long a yellow-taxi trip will take (in minutes) from what is known when it starts. This is a regression problem.

**Data:** NYC TLC Yellow Taxi Trip Records, January to April 2026, in `OriginalData_DoNotDelete/`
(source: <https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page>). Column meanings are in [Variables.md](Variables.md).

| Section | Rubric criterion |
|---|---|
| 0. Setup | 6. Reproducible notebook |
| 1. Data cleaning and preprocessing | 1. Data cleaning and preprocessing |

How the thresholds were chosen is shown with histograms in [Evidence/Distance&Duration.ipynb](Evidence/Distance&Duration.ipynb).

## 0. Setup

Every path, threshold and seed is set in the configuration cell below, so nothing further down is hard-coded.

In [1]:
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

In [2]:
# ---- Configuration: change things here, not further down ----
SEED = 4207  # cleaning has no randomness; later sections use this

# Thresholds (justified in section 1.3 and in the Evidence notebook)
DURATION_THRESHOLD = 90   # minutes; keep (0, 90]
DISTANCE_THRESHOLD = 40   # miles; keep (0, 40]
MIN_SPEED_MPH = 1         # average speed over the whole trip
MAX_SPEED_MPH = 65        # highest speed limit in New York State
INVALID_ZONE_START = 264  # zones 264 (unknown) and 265 (outside NYC)
RATECODES = [1, 2, 3, 4, 5, 6, 99]  # 99 = null/unknown in the TLC data dictionary
ZONE_IDS = list(range(1, INVALID_ZONE_START))  # the 263 real zones

# One raw file per month. The key is the month the file is meant to hold.
FILE_NAMES = {
    "2026-01": "yellow_tripdata_2026-Jan",
    "2026-02": "yellow_tripdata_2026-Feb",
    "2026-03": "yellow_tripdata_2026-Mar",
    "2026-04": "yellow_tripdata_2026-Apr",
}

# Split by time, never at random: train on the past, test on the future
SPLIT = {
    "train": ["2026-01", "2026-02"],
    "validation": ["2026-03"],
    "test": ["2026-04"],
}

PICKUP, DROPOFF = "tpep_pickup_datetime", "tpep_dropoff_datetime"

# Only the columns cleaning needs. The other 10 columns are never read from disk.
COLUMNS = [PICKUP, DROPOFF, "trip_distance", "PULocationID", "DOLocationID",
           "VendorID", "RatecodeID", "payment_type", "fare_amount"]

np.random.seed(SEED)
pd.set_option("display.width", 200)

In [3]:
# Works out the project root, so the notebook runs from any folder on anyone's machine
def find_project_root(marker="OriginalData_DoNotDelete", start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"Could not find '{marker}' above {start}")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "OriginalData_DoNotDelete"
CLEANED_DIR = PROJECT_ROOT / "CleanedData"
CLEANED_DIR.mkdir(exist_ok=True)

# Record the environment so anyone can reproduce the run
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | pyarrow {pa.__version__} | numpy {np.__version__}")
print(f"{platform.platform()} | {os.cpu_count()} CPU cores")
NOTEBOOK_START = time.perf_counter()

Python 3.14.8 | pandas 3.0.6 | pyarrow 26.0.1 | numpy 2.5.3
Windows-11-10.0.26200-SP0 | 14 CPU cores


## 1. Data cleaning and preprocessing

This section is sequential. The work is split into small functions that handle one month at a time
(`read_month`, `clean_data`, `preprocess`), so the data-parallel version can later run the same functions on each month in parallel.

### 1.1 Inspect the Parquet footers first

A Parquet file keeps its row counts and each column's min, max and null count in a small footer at the end of the file.
Reading only the footer shows us the size of the data and most of its problems without loading any rows.

In [4]:
# Combine the per-row-group statistics in one file's footer into one min / max / null count per column
def footer_stats(path):
    meta = pq.ParquetFile(path).metadata
    stats = {}
    for rg in range(meta.num_row_groups):
        for ci in range(meta.num_columns):
            col = meta.row_group(rg).column(ci)
            s = col.statistics
            if s is None:
                continue
            d = stats.setdefault(col.path_in_schema, {"min": None, "max": None, "nulls": 0})
            d["nulls"] += s.null_count or 0
            if s.has_min_max:
                d["min"] = s.min if d["min"] is None else min(d["min"], s.min)
                d["max"] = s.max if d["max"] is None else max(d["max"], s.max)
    return meta, stats


FOOTER, rows = {}, []
for month, name in FILE_NAMES.items():
    meta, stats = footer_stats(DATA_DIR / f"{name}.parquet")
    FOOTER[month] = stats
    rows.append({"month": month, "rows": meta.num_rows, "row_groups": meta.num_row_groups,
                 "first_pickup": stats[PICKUP]["min"], "last_pickup": stats[PICKUP]["max"],
                 "max_distance": stats["trip_distance"]["max"], "min_fare": stats["fare_amount"]["min"],
                 "null_RatecodeID": stats["RatecodeID"]["nulls"]})

RAW_ROWS = {r["month"]: r["rows"] for r in rows}
print(f"Total raw rows: {sum(RAW_ROWS.values()):,}")
pd.DataFrame(rows).set_index("month")

Total raw rows: 14,908,446


,rows,row_groups,first_pickup,last_pickup,max_distance,min_fare,null_RatecodeID
month,,,,,,,
2026-01,3724889,4,2025-12-31 23:57:29,2026-02-01 00:45:01,269097.48,-2555.2,1088058
2026-02,3399866,4,2026-01-31 23:31:23,2026-03-01 00:51:48,328522.20,-2084.1,1023317
2026-03,3952451,4,2008-12-31 23:03:20,2026-04-01 00:06:25,288381.68,-990.0,945748
2026-04,3831240,4,2001-01-01 09:23:58,2026-05-01 00:01:28,281576.08,-1274.9,799786


The footers already show what needs cleaning:

- **Spill-over pickups:** every file has pickups from the month before or after (`first_pickup`, `last_pickup`).
- **Meter errors:** the longest trip is hundreds of thousands of miles (`max_distance`).
- **Negative fares** (`min_fare`).
- **Flex Fare nulls:** about a quarter of rows have no `RatecodeID` (`null_RatecodeID`).

### 1.2 Load only what we need

We read 9 of the 20 columns. The spill-over filter is passed to the Parquet reader, so rows from the wrong month are dropped while the file is read and never reach pandas.

`fare_amount`, `payment_type` and the drop-off time are loaded only for cleaning and are dropped before saving (see 1.6).

In [5]:
def month_bounds(month):
    start = pd.Timestamp(f"{month}-01")
    return start, start + pd.offsets.MonthBegin(1)


# Reads one month. The filter keeps only pickups inside that month (removes spill-over).
def read_month(month):
    start, end = month_bounds(month)
    us = pa.timestamp("us")  # the files store timestamps in microseconds
    table = pq.read_table(
        DATA_DIR / f"{FILE_NAMES[month]}.parquet",
        columns=COLUMNS,
        filters=[(PICKUP, ">=", pa.scalar(start.to_pydatetime(), us)),
                 (PICKUP, "<", pa.scalar(end.to_pydatetime(), us))],
    )
    return table.to_pandas()

### 1.3 Cleaning rules

The rules run in this order. Each removed row is counted against the first rule that catches it, so the counts add up exactly.

| Rule | Removed when | Why |
|---|---|---|
| Spill-over | pickup is not in the file's month | Records from the month before or after. They would also land on the wrong side of the time-based split. |
| Duration | duration ≤ 0 or > 90 min | Drop-off at or before pickup is impossible. Over 90 min is mostly meters left running (max 5.2 days); 99% of trips are under 72 min. |
| Distance | distance ≤ 0 or > 40 miles | Zero means no movement or a meter fault. Over 40 miles removes meter errors such as 269,097 miles; 99% of trips are under 20 miles. |
| Zones | pickup or drop-off zone is 264 or 265 | "Unknown" and "outside NYC" have no usable location. |
| Speed | average speed < 1 mph or > 65 mph | Catches meter errors that pass the distance and duration rules on their own, e.g. 30 miles in 2 minutes. |
| Fare | fare ≤ 0 | Refunds, disputes and voided trips, not real journeys. Most negative fares are a reversal of another record for the same trip. |
| Duplicate | same vendor, pickup, drop-off, zones and distance as an earlier row | The same trip recorded twice. The first copy is kept. |

The fare is used only to spot bad records. It is never a feature, because the meter calculates it from the trip's time and distance.

In [6]:
DUPLICATE_KEY = ["VendorID", PICKUP, DROPOFF, "PULocationID", "DOLocationID", "trip_distance"]


# returns string for what data was and is currently and what amount was dropped
def report(before, after):
    return (f"Previous Dataset Size: {before:,} \n"
            f"Current Dataset Size: {after:,} \n"
            f"Dropped: {before - after:,} lines ({1 - after / before:.2%})")


# Applies each rule in order and counts how many rows each one removes
def clean_data(dataframe_original):
    dataframe = dataframe_original.copy()
    dataframe["duration_min"] = (dataframe[DROPOFF] - dataframe[PICKUP]).dt.total_seconds() / 60

    steps = [
        ("Duration", lambda d: (d["duration_min"] > 0) & (d["duration_min"] <= DURATION_THRESHOLD)),
        ("Distance", lambda d: d["trip_distance"].between(0, DISTANCE_THRESHOLD, inclusive="right")),
        ("Zones 264/265", lambda d: (d["PULocationID"] < INVALID_ZONE_START) & (d["DOLocationID"] < INVALID_ZONE_START)),
        ("Speed", lambda d: (d["trip_distance"] / (d["duration_min"] / 60)).between(MIN_SPEED_MPH, MAX_SPEED_MPH)),
        ("Fare", lambda d: d["fare_amount"] > 0),
        ("Duplicate", lambda d: ~d.duplicated(DUPLICATE_KEY, keep="first")),
    ]

    removed = {}
    for name, keep in steps:
        before = len(dataframe)
        dataframe = dataframe.loc[keep(dataframe)]
        removed[name] = before - len(dataframe)
    return dataframe, removed

### 1.4 Preprocessing: Flex Fare, categories and features

**Flex Fare nulls.** Rows with `payment_type = 0` are Flex Fare trips, booked in an app at an up-front price.
For these rows the vendors leave `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` and `Airport_fee` empty
(section 1.7 checks this). They are about a quarter of all trips and their times, zones and distances are fine, so we keep them instead of dropping every null:

- `RatecodeID` nulls become `99`, the data dictionary's code for "null/unknown".
- A new column `is_flex_fare` records which trips were Flex Fare. This is known when the trip is booked, so it is safe to use as a feature.
- The other null columns are never loaded.

**Categories.** Zone IDs and rate codes are labels, not quantities (zone 200 is not "twice" zone 100), so they are stored as categories
with a fixed list of values. Every month then gets the same codes, whichever order the months are processed in.

**Features known at pickup.** `pickup_hour` (0 to 23) and `pickup_dow` (0 = Monday to 6 = Sunday).

In [7]:
# Output columns, in a fixed order. duration_min is the target.
CLEAN_COLUMNS = [PICKUP, "pickup_hour", "pickup_dow", "PULocationID", "DOLocationID",
                 "trip_distance", "RatecodeID", "is_flex_fare", "duration_min"]


def preprocess(dataframe):
    out = pd.DataFrame({
        PICKUP: dataframe[PICKUP],
        "pickup_hour": dataframe[PICKUP].dt.hour.astype("int8"),
        "pickup_dow": dataframe[PICKUP].dt.dayofweek.astype("int8"),
        "PULocationID": pd.Categorical(dataframe["PULocationID"], categories=ZONE_IDS),
        "DOLocationID": pd.Categorical(dataframe["DOLocationID"], categories=ZONE_IDS),
        "trip_distance": dataframe["trip_distance"],
        "RatecodeID": pd.Categorical(dataframe["RatecodeID"].fillna(99).astype("int64"), categories=RATECODES),
        "is_flex_fare": dataframe["payment_type"] == 0,
        "duration_min": dataframe["duration_min"],
    })[CLEAN_COLUMNS]
    # A value missing from a category list would quietly turn into NaN, so stop if that happens
    assert not out.isna().any().any(), "unexpected null after preprocessing"
    return out.reset_index(drop=True)


# Read a cleaned month back with its categories restored (Parquet does not always keep them)
def read_clean(month, columns=None):
    dataframe = pd.read_parquet(CLEANED_DIR / f"{FILE_NAMES[month]}_Cleaned.parquet", columns=columns)
    for col, cats in [("PULocationID", ZONE_IDS), ("DOLocationID", ZONE_IDS), ("RatecodeID", RATECODES)]:
        if col in dataframe:
            dataframe[col] = pd.Categorical(dataframe[col], categories=cats)
    return dataframe

### 1.5 Run the cleaning and report before and after row counts

Each month is read, cleaned, preprocessed and saved to `CleanedData/` on its own, so only one month is in memory at a time.
The spill-over count is the footer row count minus the rows the filtered read returned.

In [8]:
removed_by_month, kept_by_month = {}, {}
start = time.perf_counter()

for month in FILE_NAMES:
    raw = read_month(month)
    removed = {"Spill-over": RAW_ROWS[month] - len(raw)}
    clean, removed_by_rule = clean_data(raw)
    removed.update(removed_by_rule)

    out = preprocess(clean)
    out.to_parquet(CLEANED_DIR / f"{FILE_NAMES[month]}_Cleaned.parquet", index=False)

    removed_by_month[month], kept_by_month[month] = removed, len(out)
    print(f"--- {month} ---\n{report(RAW_ROWS[month], len(out))}\n")
    del raw, clean, out  # free memory before the next month

print(f"Cleaning took {time.perf_counter() - start:.1f}s")

--- 2026-01 ---
Previous Dataset Size: 3,724,889 
Current Dataset Size: 3,454,521 
Dropped: 270,368 lines (7.26%)

--- 2026-02 ---
Previous Dataset Size: 3,399,866 
Current Dataset Size: 3,150,911 
Dropped: 248,955 lines (7.32%)

--- 2026-03 ---
Previous Dataset Size: 3,952,451 
Current Dataset Size: 3,700,191 
Dropped: 252,260 lines (6.38%)

--- 2026-04 ---
Previous Dataset Size: 3,831,240 
Current Dataset Size: 3,618,002 
Dropped: 213,238 lines (5.57%)

Cleaning took 15.0s


In [9]:
# One table: rows removed by each rule, per month and in total
rules = list(next(iter(removed_by_month.values())))
summary = pd.DataFrame(removed_by_month).loc[rules]
summary.loc["Rows before"] = pd.Series(RAW_ROWS)
summary.loc["Rows removed"] = summary.loc[rules].sum()
summary.loc["Rows after"] = pd.Series(kept_by_month)
summary = summary.loc[["Rows before", *rules, "Rows removed", "Rows after"]]
summary["Total"] = summary.sum(axis=1)
summary["% of raw"] = (100 * summary["Total"] / summary.loc["Rows before", "Total"]).round(2)

# Every raw row is either kept or removed by exactly one rule
assert (summary.loc["Rows before"] - summary.loc["Rows removed"] == summary.loc["Rows after"]).all()

summary.to_csv(CLEANED_DIR / "cleaning_report.csv")
summary

,2026-01,2026-02,2026-03,2026-04,Total,% of raw
Rows before,3724889,3399866,3952451,3831240,14908446,100.00
Spill-over,7,16,19,12,54,0.00
Duration,60149,54482,67141,67638,249410,1.67
Distance,125288,123080,120601,93615,462584,3.10
Zones 264/265,22266,19140,20714,20756,82876,0.56
Speed,25548,26551,23257,16541,91897,0.62
Fare,37110,25686,20528,14676,98000,0.66
Duplicate,0,0,0,0,0,0.00
Rows removed,270368,248955,252260,213238,984821,6.61
Rows after,3454521,3150911,3700191,3618002,13923625,93.39


The duplicate rule removes nothing because the fare rule has already dealt with duplicates. In January's raw data, 35,174 of the 35,713 duplicate pairs
are a trip plus its negative-fare reversal, and the rest are caught by the duration or distance rules. We keep the rule as a safety check.

### 1.6 Check the Flex Fare nulls

If the nulls come from Flex Fare, the number of `payment_type = 0` rows should match the null count of each affected column exactly.
The null counts come from the footers, so this only needs to read one column.

In [10]:
flex_rows = []
for month, name in FILE_NAMES.items():
    payment = pq.read_table(DATA_DIR / f"{name}.parquet", columns=["payment_type"])["payment_type"]
    row = {"month": month, "flex_fare_rows": pc.sum(pc.equal(payment, 0)).as_py()}
    for col in ["passenger_count", "RatecodeID", "store_and_fwd_flag", "congestion_surcharge", "Airport_fee"]:
        row[f"nulls_{col}"] = FOOTER[month][col]["nulls"]
    flex_rows.append(row)

flex = pd.DataFrame(flex_rows).set_index("month")
assert all((flex[c] == flex["flex_fare_rows"]).all() for c in flex.columns if c.startswith("nulls_"))
print("The nulls are exactly the Flex Fare rows in every month.")
flex

The nulls are exactly the Flex Fare rows in every month.


,flex_fare_rows,nulls_passenger_count,nulls_RatecodeID,nulls_store_and_fwd_flag,nulls_congestion_surcharge,nulls_Airport_fee
month,,,,,,
2026-01,1088058,1088058,1088058,1088058,1088058,1088058
2026-02,1023317,1023317,1023317,1023317,1023317,1023317
2026-03,945748,945748,945748,945748,945748,945748
2026-04,799786,799786,799786,799786,799786,799786


In [11]:
# Share of Flex Fare trips that survived cleaning, and how their durations compare
flex_after = pd.concat([read_clean(m, ["is_flex_fare", "duration_min"]) for m in FILE_NAMES])
flex_after.groupby("is_flex_fare")["duration_min"].agg(trips="size", median_minutes="median").round(2)

,trips,median_minutes
is_flex_fare,,
False,10492869,12.77
True,3430756,16.18


### 1.7 Train, validation and test split by time

An ETA model predicts future trips, so we split by month instead of at random:
**train = January and February, validation = March, test = April.**
A random split puts trips from the same hour and the same traffic into both train and test, which makes the model look better than it is.
Validation is for choosing models and hyperparameters; the test month is used once, at the end.

In [12]:
# Load one split from the cleaned files
def load_split(name, columns=None):
    return pd.concat([read_clean(m, columns) for m in SPLIT[name]], ignore_index=True)


split_rows = []
for name in SPLIT:
    pickups = load_split(name, [PICKUP])[PICKUP]
    split_rows.append({"split": name, "months": ", ".join(SPLIT[name]), "rows": len(pickups),
                       "first_pickup": pickups.min(), "last_pickup": pickups.max()})

splits = pd.DataFrame(split_rows).set_index("split")
splits["share_%"] = (100 * splits["rows"] / splits["rows"].sum()).round(1)

# The three splits must not overlap in time
assert splits.loc["train", "last_pickup"] < splits.loc["validation", "first_pickup"]
assert splits.loc["validation", "last_pickup"] < splits.loc["test", "first_pickup"]
splits

,months,rows,first_pickup,last_pickup,share_%
split,,,,,
train,"2026-01, 2026-02",6605432,2026-01-01,2026-02-28 23:59:59,47.4
validation,2026-03,3700191,2026-03-01,2026-03-31 23:59:59,26.6
test,2026-04,3618002,2026-04-01,2026-04-30 23:59:59,26.0


### 1.8 Leakage check

| Risk | How we handle it |
|---|---|
| The drop-off time gives away the answer | Used once to work out `duration_min`, then dropped. |
| Fares are calculated from time and distance after the trip | `fare_amount` is used only to remove bad records, then dropped. The other money columns are never loaded. |
| `payment_type` codes such as "dispute" are only known after the trip | Reduced to `is_flex_fare`, which is known at booking. |
| Test data influencing training | Split by time. The thresholds are fixed numbers, not statistics fitted to the data. |
| Average-time features (e.g. borough to borough by hour) | Not built yet. When they are, they must be worked out from the training months only. |
| `trip_distance` is measured by the meter during the trip | Kept as a stand-in for the planned route distance a real ETA system would get from a map. We say this on the poster. |

In [13]:
LEAKAGE_COLUMNS = {DROPOFF, "fare_amount", "payment_type", "tip_amount", "total_amount", "tolls_amount",
                   "extra", "mta_tax", "improvement_surcharge", "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"}

for month, name in FILE_NAMES.items():
    saved = set(pq.read_schema(CLEANED_DIR / f"{name}_Cleaned.parquet").names)
    assert saved == set(CLEAN_COLUMNS) and not saved & LEAKAGE_COLUMNS
print("No leakage columns in the cleaned files.")

sample = read_clean("2026-01")
display(sample.head())
display(sample.dtypes.to_frame("dtype").T)
sample[["trip_distance", "duration_min"]].describe().T.round(2)

No leakage columns in the cleaned files.


,tpep_pickup_datetime,pickup_hour,pickup_dow,PULocationID,DOLocationID,trip_distance,RatecodeID,is_flex_fare,duration_min
0,2026-01-01 00:54:04,0,3,239,238,0.97,1,False,5.550000
1,2026-01-01 00:34:04,0,3,163,162,0.90,1,False,5.716667
2,2026-01-01 00:57:06,0,3,43,237,1.40,1,False,8.883333
3,2026-01-01 00:15:22,0,3,142,209,5.58,1,False,42.800000
4,2026-01-01 00:27:13,0,3,88,144,2.16,1,False,13.500000


,tpep_pickup_datetime,pickup_hour,pickup_dow,PULocationID,DOLocationID,trip_distance,RatecodeID,is_flex_fare,duration_min
dtype,datetime64[us],int8,int8,category,category,float64,category,bool,float64


,count,mean,std,min,25%,50%,75%,max
trip_distance,3454521.0,3.41,3.99,0.01,1.10,1.90,3.81,40.0
duration_min,3454521.0,16.68,12.49,0.03,8.28,13.42,21.08,90.0


In [14]:
print(f"Section 1 run time: {time.perf_counter() - NOTEBOOK_START:.1f}s on {os.cpu_count()} cores")

Section 1 run time: 16.9s on 14 cores
